# Сколько раз встретился ФП 1.2 по логике двух отчетов

Тетрадка считает количество ФП `1.2` (только `TYPE_FP = ФП`) по двум методикам:
1. Логика `analysis_crm_segments.ipynb`.
2. Логика `analysis_scenario_ipu_results.ipynb` (двухпоточная дедупликация scenario/ipu).

На выходе формируется сравнение в таблицах и выгрузка в Excel.

In [ ]:
import re
import unicodedata
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 200)

DATE_FROM = pd.Timestamp("2023-01-01")
DATE_TO = pd.Timestamp("2025-12-31")
ALLOWED_SOURCES = ["Н2.0", "Справочник CRM-системы", "CRM-система"]
SEGMENT_MAP = {
    "ДМСБ (микро)": "МкБ",
    "ДМСБ (малый)": "МСБ",
    "ДМСБ (средний)": "МСБ",
    "ДКБ": "ККБ",
}
DEDUP_KEY = ["inn", "fp_num", "fp_type", "IDENTIFICATION_DATE"]

SCR_DATE_COLS = ["END_DATE_SCR_FCT", "END_DATE_SCR_PLAN"]
IPU_DATE_COLS = ["FIRST_END_DATE_EVENT", "NEW_PLAN_END_DATE_EVT", "END_EVENT_DATE_FACT"]

_LAT2CYR = str.maketrans({
    "A": "А", "B": "В", "C": "С", "E": "Е", "H": "Н", "K": "К",
    "M": "М", "O": "О", "P": "Р", "T": "Т", "X": "Х",
    "a": "а", "c": "с", "e": "е", "o": "о", "p": "р", "x": "х", "y": "у",
})


def normalize_inn(value):
    if pd.isna(value):
        return None
    s = str(value).strip()
    if s.endswith(".0"):
        s = s[:-2]
    s = s.lstrip("0") or "0"
    return s.zfill(10) if len(s) <= 10 else s.zfill(12)


def norm_text(value):
    if pd.isna(value) or value == "":
        return ""
    s = str(value)
    s = unicodedata.normalize("NFKC", s)
    s = s.translate(_LAT2CYR)
    s = re.sub(r"[\s\xa0\u200b\ufeff]+", " ", s)
    return s.strip()


# Жесткие банковые пути (без fallback).
DATA_DIR = Path("/home/jovyan/documents/DMUKP_FP_DEF/data")
CRM_FILE = DATA_DIR / "crm_last_version.csv"
OUT_XLSX = DATA_DIR / "fp_1_2_count_by_two_logics.xlsx"

print("CRM_FILE:", CRM_FILE)
print("OUT_XLSX:", OUT_XLSX)

In [ ]:
# Загрузка CSV (поддержка разных разделителей).
raw = pd.read_csv(CRM_FILE, dtype=str, low_memory=False)
if len(raw.columns) == 1 and ";" in str(raw.columns[0]):
    raw = pd.read_csv(CRM_FILE, dtype=str, low_memory=False, sep=";")

raw.columns = raw.columns.str.strip()
if "DESC_TEXT.1" in raw.columns and "DESC_TEXT_1" not in raw.columns:
    raw = raw.rename(columns={"DESC_TEXT.1": "DESC_TEXT_1"})

print("Raw rows:", len(raw))
print("Columns:", len(raw.columns))

In [ ]:
# Единая базовая подготовка (как в двух отчетах).
base = raw.copy()

if "VAL" in base.columns:
    base = base[base["VAL"].astype(str).str.strip().isin(ALLOWED_SOURCES)].copy()

base["inn"] = base["X_INN"].apply(normalize_inn)
base["dt"] = pd.to_datetime(base["IDENTIFICATION_DATE"], dayfirst=True, errors="coerce")
base = base[(base["dt"] >= DATE_FROM) & (base["dt"] <= DATE_TO)].copy()

base["segment"] = base["X_AREA_RESP"].astype(str).str.strip().map(SEGMENT_MAP)
base = base[base["segment"].notna()].copy()

base = base.dropna(subset=["inn", "NUMBER_FP_SFP"]).copy()
base["fp_num"] = base["NUMBER_FP_SFP"].astype(str).str.strip()
base["fp_type"] = base["TYPE_FP"].astype(str).str.strip().replace({"FP": "ФП", "SFP": "СФП"})

base["scenario"] = base["DESC_TEXT_1"].apply(norm_text)
base["ipu"] = base["DESC_TEXT"].apply(norm_text)

print("Base rows after common filters:", len(base))
print("Base unique DEDUP_KEY:", base.drop_duplicates(DEDUP_KEY).shape[0])

## Логика 1: `analysis_crm_segments.ipynb`

In [ ]:
crm_df = base.drop_duplicates(subset=["inn", "fp_num", "fp_type", "IDENTIFICATION_DATE"]).copy()

crm_fp12 = crm_df[(crm_df["fp_num"] == "1.2") & (crm_df["fp_type"] == "ФП")].copy()
crm_count_total = len(crm_fp12)

crm_by_segment = (
    crm_fp12.groupby("segment", as_index=False)
    .size()
    .rename(columns={"size": "count_fp_1_2"})
    .sort_values("count_fp_1_2", ascending=False)
)

print("FP 1.2 (CRM logic) total:", crm_count_total)
display(crm_by_segment)

## Логика 2: `analysis_scenario_ipu_results.ipynb`

In [ ]:
# Классификация сценариев встроена в тетрадку (автономно, без внешних файлов).
SCR_POSITIVE = [
    "ФП/СФП устранен", "ФП/СФП урегулирован", "Микро_У ФП урегулирован",
    "1_ККБ/МСБ_ФП устранен", "СФП устранен", "Микро_У Фактор устранен",
    "ФП/СФП устранен (договор закрыт)", "2_ККБ/МСБ_СФП устранен",
    "ДКБ_Принято решение УОБ об урегулировании ФП", "ФП устранен (договор закрыт)",
    "ДКБ_Не требуется решение УЛ/УОБ о снятии ФП с контроля", "Микро_У ФП устранен",
    "ФП устранен", "Микро_У СФП устранен", "ДКБ_СФП устранен", "Микро_У СФП урегулирован",
]

SCR_COND_POSITIVE = [
    "МСБ_Принято решение УОБ о снятии ФП с контроля",
    "МСБ_Принято решение УЛ о снятии ФП с контроля",
    "Микро_У Требуется принятие решения УЛ о снятии фактора с контроля",
    "Принято решение УЛ о снятии ФП с контроля",
    "ДКБ_Принято решение УЛ о снятии ФП с контроля",
    "Принято решение УОБ о снятии ФП с контроля",
    "ДКБ_Принято решение УОБ о снятии ФП с контроля",
    "Микро_У Принято решение УОБ о снятии фактора с контроля",
    "ДКБ_ФП не оказывает негативного влияния на исполнение участником кредитной сделки обязательств перед Банком, возможно снятие ФП с контроля",
    "ФП урегулирован, требуется принятие решения УОБ о снятии ФП с контроля",
    "Микро_У Принято решение УОБ о снятии ФП с контроля",
    "Микро_У Принято решение УЛ о снятии ФП с контроля",
    "ФП не устранен, не оказывает негативного влияния, требует рассмотрения вопроса на УОБ с целью снятия ФП с контроля",
    "11_ККБ/МСБ_ФП не устранен в рамках сценария, не оказывает негативного влияния на исполнение участником кредитной сделки обязательств перед Банком, (требуется решение УОБ о снятии ФП с контроля)",
    "4_МСБ_Право Банка реализовано, ФП снят с контроля",
    "5_МСБ_Право Банка не реализовано, ФП снят с контроля",
    "Микро_У ФП не устранен, не оказывает негативного влияния на исполнение участником кредитной сделки обязательств перед Банком",
    "5_ККБ/МСБ_ФП не требует устранения, право Банка не реализовано",
    "ФП не устранен, не оказывает негативного влияния на исполнение участником кредитной сделки обязательств перед Банком",
    "ФП не устранен в рамках сценария, не оказывает негативного влияния на исполнение участником кредитной сделки обязательств перед Банком, (требуется решение УОБ о снятии ФП с контроля)",
    "4_ККБ/МСБ_ФП не требует устранения, право Банка реализовано",
    "ФП не требует устранения, право Банка не реализовано",
    "Микро_У Фактор устранен - решение УОБ о снятии фактора с контроля",
    "Микро_У Принятие решения УЛ о снятии фактора с контроля",
    "Микро_У Требуется принятие решения УОБ о снятии фактора с контроля",
    "Микро_У Требуется принятие решения УОБ/УЛ о снятии ФП с контроля",
]

SCR_NEGATIVE = [
    "ФП/СФП не устранен, оказывает негативное влияние, требует рассмотрения на УОБ вопроса о признании задолженности проблемной",
    "3_ККБ/МСБ_СФП не устранен, требует рассмотрения на УОБ вопроса о признании задолженности проблемной",
    "Микро_У Фактор не устранен, оказывает негативное влияние, требует рассмотрения на УОБ вопроса о признании задолженности проблемной",
    "ДКБ_СФП не устранен, требует рассмотрения на УОБ вопроса о признании задолженности проблемной",
    "12_ККБ/МСБ_ФП не устранен, оказывает негативное влияние, требует рассмотрения на УОБ вопроса о признании задолженности проблемной",
    "Микро_У СФП не устранен, оказывает негативное влияние, требует рассмотрения на УОБ вопроса о признании задолженности проблемной",
    "ДКБ_ФП оказывает негативное влияние, требует рассмотрения УОБ вопроса о признании задолженности проблемной",
]

SCR_COND_NEGATIVE = [
    "ФП оказывает негативное влияние на исполнение участником кредитной сделки обязательств перед Банком, требуется разработка ПУ",
    "ФП не устранен в рамках сценария, требуется разработка ИПУ",
    "Микро_У ФП не устранен в рамках сценария, требуется разработка ИПУ",
    "10_ККБ/МСБ_ФП не устранен в рамках сценария, оказывает негативное влияние на исполнение участником кредитной сделки обязательств перед Банком, требуется разработка ИПУ",
    "ФП не устранен в рамках сценария, оказывает негативное влияние, требует рассмотрения вопроса на УОБ об утверждении Плана устранения ФП",
    "ФП не устранен. Лимит снижен до \"0\". Лимит восстановлению не подлежит.",
    "Микро_У Техническое закрытие ФП в связи с запуском нового сценария по СФП (СФП 15.2У/15.2.1У)",
    "Техническое закрытие ФП, открытие нового ФП в связи с запуском нового сценария",
    "7_ККБ/МСБ_Техническое закрытие ФП, открытие нового ФП в связи с запуском нового сценария",
    "Микро_У Техническое закрытие ФП, открытие нового ФП в связи с запуском нового сценария",
    "ДКБ_Техническое закрытие ФП, открытие нового ФП в связи с запуском нового сценария",
]

SCR_ERROR = [
    "0_Информация проверена, отсутствуют основания для выявления ФП/СФП",
    "ДКБ_Информация проверена, отсутствуют основания для выявления ФП/СФП",
    "Микро_У Информация проверена, отсутствуют основания для выявления ФП/СФП",
]

SCR_ACTIVE = [
    "Активный (срок окончания, предусмотренный Порядком не наступил, находится в стадии реализации)",
    "Активный (срок окончания, предусмотренный Порядком наступил, результат реализации по состоянию на дату Реестра отсутствует)",
]


def norm_list(values: list[str]) -> list[str]:
    return [norm_text(v) for v in values]


SCR_POSITIVE = set(norm_list(SCR_POSITIVE))
SCR_COND_POSITIVE = set(norm_list(SCR_COND_POSITIVE))
SCR_NEGATIVE = set(norm_list(SCR_NEGATIVE))
SCR_COND_NEGATIVE = set(norm_list(SCR_COND_NEGATIVE))
SCR_ERROR = set(norm_list(SCR_ERROR))
SCR_ACTIVE = set(norm_list(SCR_ACTIVE))

_scr_sets = {
    "положительный": SCR_POSITIVE,
    "условно положительный": SCR_COND_POSITIVE,
    "отрицательный": SCR_NEGATIVE,
    "условно отрицательный": SCR_COND_NEGATIVE,
    "ошибка": SCR_ERROR,
    "активный": SCR_ACTIVE,
}


def classify_scr(value: str) -> str:
    for cls, items in _scr_sets.items():
        if value in items:
            return cls
    return "[пусто]" if value == "" else "неклассифицированный"


print("Embedded scenario classification loaded")
print({k: len(v) for k, v in _scr_sets.items()})

In [ ]:
s = base.copy()

for col in SCR_DATE_COLS + IPU_DATE_COLS:
    s[f"_{col}_dt"] = pd.to_datetime(s[col], dayfirst=True, errors="coerce")

scr_dt = [f"_{c}_dt" for c in SCR_DATE_COLS]
ipu_dt = [f"_{c}_dt" for c in IPU_DATE_COLS]
s["_scr_max"] = s[scr_dt].max(axis=1)
s["_ipu_max"] = s[ipu_dt].max(axis=1)

grp_has_ipu = s.groupby(DEDUP_KEY)["_ipu_max"].transform("max").notna()
stream1_raw = s[~grp_has_ipu].copy()
stream2_raw = s[grp_has_ipu].copy()

stream1_raw = stream1_raw[stream1_raw.groupby(DEDUP_KEY)["_scr_max"].transform("max").notna()]
stream1_sorted = stream1_raw.sort_values(["_END_DATE_SCR_FCT_dt", "_END_DATE_SCR_PLAN_dt"], ascending=[False, False], na_position="last")
df_stream1 = stream1_sorted.drop_duplicates(subset=DEDUP_KEY, keep="first").copy()
df_stream1["stream"] = "scenario"

stream2_raw = stream2_raw[stream2_raw.groupby(DEDUP_KEY)["_ipu_max"].transform("max").notna()]
stream2_sorted = stream2_raw.sort_values(["_END_EVENT_DATE_FACT_dt", "_NEW_PLAN_END_DATE_EVT_dt", "_FIRST_END_DATE_EVENT_dt"], ascending=[False, False, False], na_position="last")
df_stream2 = stream2_sorted.drop_duplicates(subset=DEDUP_KEY, keep="first").copy()
df_stream2["stream"] = "ipu"

sc_df = pd.concat([df_stream1, df_stream2], ignore_index=True)
sc_df["scr_class"] = sc_df["scenario"].apply(classify_scr)

# Как в отчете: исключаем [пусто] и неклассифицированный scr_class.
sc_df_valid = sc_df[(sc_df["scr_class"] != "[пусто]") & (sc_df["scr_class"] != "неклассифицированный")].copy()

sc_fp12_before = sc_df[(sc_df["fp_num"] == "1.2") & (sc_df["fp_type"] == "ФП")].copy()
sc_fp12_after = sc_df_valid[(sc_df_valid["fp_num"] == "1.2") & (sc_df_valid["fp_type"] == "ФП")].copy()

sc_counts = pd.DataFrame([
    {"metric": "fp_1_2_before_scr_filter", "count": len(sc_fp12_before)},
    {"metric": "fp_1_2_after_scr_filter", "count": len(sc_fp12_after)},
])

sc_by_segment = (
    sc_fp12_after.groupby("segment", as_index=False)
    .size()
    .rename(columns={"size": "count_fp_1_2"})
    .sort_values("count_fp_1_2", ascending=False)
)

display(sc_counts)
display(sc_by_segment)

## Сводное сравнение двух логик

In [ ]:
# Доп. метрика по запросу: логика сценарного отчета без разделения на stream1/stream2.
scenario_base_fp12 = (
    base.drop_duplicates(DEDUP_KEY)
    .loc[lambda d: (d["fp_num"] == "1.2") & (d["fp_type"] == "ФП")]
    .copy()
)

scenario_base_by_segment = (
    scenario_base_fp12.groupby("segment", as_index=False)
    .size()
    .rename(columns={"size": "count_scenario_base_no_stream_split"})
    .sort_values("count_scenario_base_no_stream_split", ascending=False)
)

summary_total = pd.DataFrame([
    {"logic": "analysis_crm_segments", "count_fp_1_2": len(crm_fp12)},
    {"logic": "analysis_scenario_base_no_stream_split", "count_fp_1_2": len(scenario_base_fp12)},
    {"logic": "analysis_scenario_ipu_results_before_scr_filter", "count_fp_1_2": len(sc_fp12_before)},
    {"logic": "analysis_scenario_ipu_results_after_scr_filter", "count_fp_1_2": len(sc_fp12_after)},
])

crm_seg = crm_by_segment.rename(columns={"count_fp_1_2": "count_crm_logic"})
base_seg = scenario_base_by_segment.rename(columns={"count_scenario_base_no_stream_split": "count_scenario_base_no_stream_split"})
sc_seg = sc_by_segment.rename(columns={"count_fp_1_2": "count_scenario_ipu_logic"})

summary_by_segment = crm_seg.merge(base_seg, on="segment", how="outer").merge(sc_seg, on="segment", how="outer").fillna(0)
summary_by_segment[["count_crm_logic", "count_scenario_base_no_stream_split", "count_scenario_ipu_logic"]] = summary_by_segment[["count_crm_logic", "count_scenario_base_no_stream_split", "count_scenario_ipu_logic"]].astype(int)
summary_by_segment["delta_crm_minus_scenario_base"] = summary_by_segment["count_crm_logic"] - summary_by_segment["count_scenario_base_no_stream_split"]
summary_by_segment["delta_crm_minus_scenario_ipu"] = summary_by_segment["count_crm_logic"] - summary_by_segment["count_scenario_ipu_logic"]

display(summary_total)
display(summary_by_segment)

In [ ]:
with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
    summary_total.to_excel(writer, sheet_name="summary_total", index=False)
    summary_by_segment.to_excel(writer, sheet_name="summary_by_segment", index=False)

    crm_fp12.to_excel(writer, sheet_name="crm_logic_fp1_2_cards", index=False)
    scenario_base_fp12.to_excel(writer, sheet_name="scenario_base_fp1_2_cards", index=False)
    scenario_base_by_segment.to_excel(writer, sheet_name="scenario_base_by_segment", index=False)

    sc_fp12_before.to_excel(writer, sheet_name="scenario_ipu_fp1_2_before", index=False)
    sc_fp12_after.to_excel(writer, sheet_name="scenario_ipu_fp1_2_after", index=False)

print(f"Сохранено: {OUT_XLSX}")